# Single-leg tight-z mean-reversion taker

This small research test uses **one CSV**. It computes a causal rolling z-score of the top-of-book mid-price. It buys only when the z-score **crosses below** the negative entry signal and sells only when it **crosses above** the positive entry signal; execution is at the next snapshot's ask or bid. The position closes aggressively when the z-score crosses the mean-reversion exit signal, or at a wider stop.

The entry threshold is intentionally tight (`0.75`) to produce frequent signals in the short sample. Signals use only past observations and execute on the following snapshot. This is a simplified research proxy—not the supported foundation production replay or economic evidence.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
repo = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "data").is_dir())
CSV_NUMBER = 1  # change to 2 to test the other CSV independently
path = repo / "data" / f"market_data_2024-01-25_{CSV_NUMBER}.csv"
cols = ["exchtime", "bidpx0", "bidvol0", "askpx0", "askvol0"]
book = pd.read_csv(path, usecols=cols, parse_dates=["exchtime"])
book = book.drop_duplicates("exchtime", keep="last").sort_values("exchtime")
book = book.query("bidpx0 > 0 and askpx0 >= bidpx0 and bidvol0 > 0 and askvol0 > 0").set_index("exchtime")
book["mid"] = (book.bidpx0 + book.askpx0) / 2
print(path.name, len(book), "snapshots")
book.head()

In [ ]:
LOOKBACK = 20
ENTRY_Z = 0.75       # deliberately tight
EXIT_Z = 0.10        # close after returning very near the mean
STOP_Z = 2.50
FEE_PER_EXECUTION = 0.0
MULTIPLIER = 1.0

# shift(1) keeps the rolling reference distribution strictly historical.
history = book.mid.shift(1).rolling(LOOKBACK, min_periods=LOOKBACK)
book["zscore"] = (book.mid - history.mean()) / history.std(ddof=0)
book[["mid", "zscore"]].dropna().head()

In [ ]:
def run_mean_reversion_taker(df):
    trades, position = [], None
    clean = df.dropna(subset=["zscore"])

    for i in range(2, len(clean)):
        signal = clean.iloc[i - 1]       # crossing completes here
        prior_signal = clean.iloc[i - 2] # prior side of the crossing
        row = clean.iloc[i]              # next-snapshot taker fill
        ts = clean.index[i]

        if position is None:
            if i == len(clean) - 1:
                break
            crossed_down = prior_signal.zscore > -ENTRY_Z and signal.zscore <= -ENTRY_Z
            crossed_up = prior_signal.zscore < ENTRY_Z and signal.zscore >= ENTRY_Z
            side = 1 if crossed_down else (-1 if crossed_up else 0)
            if side:
                position = {"side": side, "entry_time": ts, "entry_z": signal.zscore,
                            "entry_price": row.askpx0 if side == 1 else row.bidpx0}
            continue

        side = position["side"]
        reverted = (side == 1 and prior_signal.zscore < -EXIT_Z and signal.zscore >= -EXIT_Z) or (side == -1 and prior_signal.zscore > EXIT_Z and signal.zscore <= EXIT_Z)
        stopped = (side == 1 and signal.zscore <= -STOP_Z) or (side == -1 and signal.zscore >= STOP_Z)
        last_row = i == len(clean) - 1
        if reverted or stopped or last_row:
            exit_price = row.bidpx0 if side == 1 else row.askpx0
            gross = side * (exit_price - position["entry_price"]) * MULTIPLIER
            trades.append({**position, "exit_time": ts, "exit_z": signal.zscore,
                           "exit_price": exit_price,
                           "exit_reason": "eod" if last_row else ("stop" if stopped else "mean_reversion"),
                           "gross_pnl": gross, "net_pnl": gross - 2 * FEE_PER_EXECUTION})
            position = None

    result = pd.DataFrame(trades)
    if not result.empty:
        result["direction"] = np.where(result.side == 1, "long", "short")
        result["holding_seconds"] = (result.exit_time - result.entry_time).dt.total_seconds()
        result["equity"] = result.net_pnl.cumsum()
    return result

trades = run_mean_reversion_taker(book)
trades

In [ ]:
if trades.empty:
    print("No completed trades. Inspect the signal or reduce ENTRY_Z.")
else:
    summary = pd.Series({
        "trades": len(trades),
        "longs": (trades.side == 1).sum(),
        "shorts": (trades.side == -1).sum(),
        "win_rate": (trades.net_pnl > 0).mean(),
        "total_net_pnl": trades.net_pnl.sum(),
        "average_net_pnl": trades.net_pnl.mean(),
        "average_holding_seconds": trades.holding_seconds.mean(),
    })
    display(summary.to_frame("value"))
    display(trades.groupby(["direction", "exit_reason"]).net_pnl.agg(["count", "sum", "mean"]))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
book.zscore.plot(ax=axes[0], color="navy", lw=0.9, title=f"{path.name}: tight causal z-score")
for level, color in [(ENTRY_Z, "crimson"), (-ENTRY_Z, "crimson"), (EXIT_Z, "gray"), (-EXIT_Z, "gray")]:
    axes[0].axhline(level, color=color, ls="--", lw=0.8)
axes[0].set_ylabel("z-score")
if not trades.empty:
    trades.set_index("exit_time").equity.plot(ax=axes[1], drawstyle="steps-post", color="darkgreen", title="Realized net P&L")
else:
    axes[1].text(0.5, 0.5, "No completed trades", ha="center", va="center", transform=axes[1].transAxes)
axes[1].set_ylabel("P&L")
plt.tight_layout();

Before treating the result as economic, set the actual multiplier and fees and model latency, depth, partial fills, session boundaries, and out-of-sample parameter selection. A tight threshold can trade noise frequently, so bid/ask crossing costs are especially important.

## Complete trade list

In [ ]:
trade_columns = ["direction", "entry_time", "entry_z", "entry_price",
                 "exit_time", "exit_z", "exit_price", "exit_reason",
                 "holding_seconds", "gross_pnl", "net_pnl", "equity"]
if trades.empty:
    print("No completed trades.")
else:
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 220):
        display(trades[trade_columns].reset_index(drop=True).rename_axis("trade_number").set_axis(range(1, len(trades) + 1)))